# Approximate inference: Gibbs sampling and particle filtering

_Artificial Intelligence — more_

**Too big to compute exactly? Draw many samples and let the histogram be your answer.**

Exact inference is too slow on big, tangled networks. Approximate inference trades exactness for speed by sampling.

     Gibbs sampling walks through the variables, resampling one at a time from its conditional given all the others. The visited states form samples from the posterior.

---

This notebook is a practice scaffold for the **Approximate inference: Gibbs sampling and particle filtering** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# unnormalized joint over (X,Y), each in {0,1}: a 2x2 table.
phi = np.array([[1.0, 2.0],
                [3.0, 4.0]])
true_post = phi / phi.sum()

def cond(fix_axis, fix_val):
    row = phi[fix_val, :] if fix_axis == 0 else phi[:, fix_val]
    return row / row.sum()

x, y = 0, 0
counts = np.zeros((2, 2))
for t in range(20000):
    x = rng.choice(2, p=cond(1, y))   # resample X | Y
    y = rng.choice(2, p=cond(0, x))   # resample Y | X
    counts[x, y] += 1

emp = counts / counts.sum()
print("true posterior:\n", np.round(true_post, 3))
print("Gibbs estimate:\n", np.round(emp, 3))
print("max abs error:", round(float(np.max(np.abs(emp - true_post))), 4))

## Visualize the data & results

_Sprinkler-Rain network: does Gibbs sampling recover the joint posterior over (Sprinkler, Rain) given wet grass?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# Same Sprinkler-Rain-WetGrass net; estimate P(Sprinkler,Rain | WetGrass=True).
P_C = np.array([0.5, 0.5])
P_S_C = np.array([[0.5, 0.5], [0.9, 0.1]])
P_R_C = np.array([[0.8, 0.2], [0.2, 0.8]])
P_W = np.zeros((2, 2, 2))
P_W[0,0] = [1.0, 0.0];  P_W[0,1] = [0.1, 0.9]
P_W[1,0] = [0.1, 0.9];  P_W[1,1] = [0.01, 0.99]
norm = lambda v: v / v.sum()

C = S = R = 0
counts = np.zeros((2, 2))
for t in range(40000):
    C = rng.choice(2, p=norm(np.array([P_C[c]*P_S_C[c,S]*P_R_C[c,R] for c in range(2)])))
    S = rng.choice(2, p=norm(np.array([P_S_C[C,s]*P_W[s,R,1] for s in range(2)])))
    R = rng.choice(2, p=norm(np.array([P_R_C[C,r]*P_W[S,r,1] for r in range(2)])))
    counts[S, R] += 1
emp = (counts / counts.sum()).flatten()

labels = ["S=off, R=no", "S=off, R=rain", "S=on, R=no", "S=on, R=rain"]
fig, ax = plt.subplots()
bars = ax.bar(labels, emp, color="#4ea1ff")
for b, v in zip(bars, emp):
    ax.text(b.get_x()+b.get_width()/2, v, str(round(v,3)), ha="center", va="bottom")
ax.set_title("Gibbs estimate of P(Sprinkler, Rain | WetGrass) (40k samples)")
plt.show()